In [ ]:
import numpy as np
raw_data = [
    [80.1, 19.8], [19.5, 20.2], [49.5, 50.2], [19.5, 80.2], [79.5, 80.2],
    [21.0, 20.5], [81.0, 20.5], [50.1, 49.8], [20.1, 79.8], [80.1, 79.8],
    [19.8, 21.2], [79.8, 21.2], [51.0, 50.5], [21.0, 80.5], [81.0, 80.5],
    [18.5, 19.5], [78.5, 19.5], [49.8, 51.2], [19.8, 81.2], [79.8, 81.2],
    [20.5, 19.0], [80.5, 19.0], [48.5, 49.5], [18.5, 79.5], [78.5, 79.5],
    [20.2, 20.8], [80.2, 20.8], [50.5, 49.0], [20.5, 79.0], [80.5, 79.0],
    [19.1, 19.9], [79.1, 19.9], [50.2, 50.8], [20.2, 80.8], [80.2, 80.8],
    [21.2, 19.3], [81.2, 19.3], [49.1, 49.9], [19.1, 79.9], [79.1, 79.9],
    [19.9, 20.1], [79.9, 20.1], [51.2, 49.3], [21.2, 79.3], [81.2, 79.3],
    [18.8, 20.9], [78.8, 20.9], [49.9, 50.1], [19.9, 80.1], [79.9, 80.1],
    [20.7, 21.1], [80.7, 21.1], [48.8, 50.9], [18.8, 80.9], [78.8, 80.9],
    [19.2, 19.2], [79.2, 19.2], [50.7, 51.1], [20.7, 81.1], [80.7, 81.1],
    [21.5, 20.4], [81.5, 20.4], [49.2, 49.2], [19.2, 79.2], [79.2, 79.2],
    [19.6, 18.9], [79.6, 18.9], [51.5, 50.4], [21.5, 80.4], [81.5, 80.4],
    [20.3, 19.6], [80.3, 19.6], [49.6, 48.9], [19.6, 78.9], [79.6, 78.9],
    [20.9, 18.8], [80.9, 18.8], [50.3, 49.6], [20.3, 79.6], [80.3, 79.6],
    [19.4, 20.7], [79.4, 20.7], [50.9, 48.8], [20.9, 78.8], [80.9, 78.8],
    [21.1, 19.7], [81.1, 19.7], [49.4, 50.7], [19.4, 80.7], [79.4, 80.7],
    [18.9, 19.1], [78.9, 19.1], [51.1, 49.7], [21.1, 79.7], [81.1, 79.7],
    [20.1, 19.8], [79.5, 20.2], [48.9, 49.1], [18.9, 79.1], [78.9, 79.1],
]
'''
(20, 20) — Bottom-Left

(80, 20) — Bottom-Right

(50, 50) — Center

(20, 80) — Top-Left

(80, 80) — Top-Right
'''
x = np.array(raw_data)
print(type(x), x.shape[0])
print(x.min())

<class 'numpy.ndarray'> 100
18.5


Initial Centroids

In [ ]:
def prob_centroid_initialization(k = "no of centroids"):
  '''
  uses probabilistic weights to find seperate initial centeroids.
  rarely fails, take average of multiple calls to get best result.
  takes 1 int arrgument:  k = "no of centroids"
  returns "2D array of centeroids"
  '''
  import random
  k = k - 1
  # Fist centeroid
  random_indices = np.random.choice(x.shape[0], size=1, replace=False)
  C = x[random_indices]

  # (x[random_indices,0] - i[0])**2 + (x[random_indices,1] - i[1])**2
  for _ in range(k):
    min_sq_dist = []
    for i in x:
      sq_dist = []
      for c in C:
        dist = (c[0] - i[0])**2 + (c[1] - i[1])**2
        sq_dist.append(dist)
      min_sq_dist.append(min(sq_dist))
    selected_point = random.choices(x, weights=min_sq_dist, k=1)[0]
    C = np.vstack([C,np.array(selected_point)])
    # print(selected_point)

  return C



Finalizing the initial centroids

In [ ]:
def final_centroids(trials = "number_of_trials",k = "no of centroids"):
  '''
  runs prob_centroid_initialization(number_of_centeroids) multiple times, picks one with lowest inertia.
  needs 2 arrgument:  trials = "number_of_trials"
                      k = "no of centroids"
  returns C as the final set of centeroids in a 2D array.
  '''
  C_list = []
  inertia = []
  for _ in range(trials):
    C = prob_centroid_initialization(k)
    I = 0
    for i in x:
      I += min(((i-C)**2).sum(axis=1))
    C_list.append(C)
    inertia.append(I)

  C = C_list[inertia.index(min(inertia))]
  return C


In [ ]:
print(final_centroids(10,5).shape[0])

5


Adjusting Centroids

In [ ]:
def adjusted_centeroids(x, C):
  '''
  finds new C , by averaginng the coordinates of all neighbouring datapoints
  takes 3 arrguments: x = "2D data", eg: [[80.1, 19.8]]
                      C = final centroids
  returns new_C
  '''
  k = C.shape[0]
  new_C = np.zeros((x[0].shape[0])*k).reshape((k, (x[0].shape[0])))
  count = np.zeros(k)

  for i in x:
    C_temp = np.square(i - C).sum(axis=1)  # find euclidian distance square
    C_min_index = np.argmin(C_temp)        # finds the index of min distance from all centeroids
    # print(C_min_index)
    count[C_min_index] += 1
    new_C[C_min_index] += i       # adds all the data points to its closest centroid group for new averaged centeroid


  for i in range(k):
    if count[i] == 0:
      count[i] = 1
    new_C[i] /= count[i]

  return new_C

### Adjusting Centroids N-times

In [ ]:
C = final_centroids(trials=10, k = 5)
for _ in range(10):
  new_C = adjusted_centeroids(x, C)
  print(new_C)
  if abs(C- new_C < 0.00004).all():
    # print("break: ", abs(C- new_C))
    break
  # print(abs(C- new_C))
  C = new_C
print(new_C)

[[20.01  19.935]
 [20.01  79.935]
 [80.01  79.935]
 [80.01  19.935]
 [50.01  49.935]]
[[20.01  19.935]
 [20.01  79.935]
 [80.01  79.935]
 [80.01  19.935]
 [50.01  49.935]]
[[20.01  19.935]
 [20.01  79.935]
 [80.01  79.935]
 [80.01  19.935]
 [50.01  49.935]]
